In [1]:
import pandas as pd
import numpy as np

df_near = pd.read_csv("../data/near_earth_asteroids_2025.csv")

df_near.rename({'q':'perihelion_distance_au', 'ad':'aphelion_distance_au', 'e':'eccentricity', 'a':'semimajor_axis_au'\
               , 'i':'inclination_deg','per':'orbital_period_days','n':'mean_motion_deg_day', 'rot_per':'rot_per_h',\
                'class':'class_code'}, axis=1, inplace=True)



/tmp/ipykernel_411689/3363829492.py:4: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df_near = pd.read_csv("../data/near_earth_asteroids_2025.csv")


In [2]:
import datetime

X_near_pha = df_near.drop(['spkid','full_name','pdes','name', 'diameter_m','moid_lunar_distances','albedo', 'rot_per_h',\
                       'moid_km', 'per_y', 'data_arc_years', 'diameter_is_estimated','pha', 'first_obs', 'last_obs'], axis=1)

X_near_pha['size_category'] = X_near_pha['size_category'].apply(lambda x: x[0:x.index(" ")].strip())
y_pha = df_near['pha']


X_near_moid = df_near.drop(['spkid','full_name','pdes','name', 'diameter_m','moid_lunar_distances','albedo','rot_per_h',\
                       'moid_km','moid_au', 'per_y', 'data_arc_years', 'diameter_is_estimated', 'first_obs', 'last_obs'], axis=1)

X_near_moid['size_category'] = X_near_moid['size_category'].apply(lambda x: x[0:x.index(" ")].strip())
y_moid = df_near['moid_au']

X_near_to_merge = df_near.drop(['spkid','pdes','name', 'diameter_m','moid_lunar_distances','albedo', 'rot_per_h',\
                       'moid_km', 'per_y', 'data_arc_years', 'diameter_is_estimated', 'first_obs', 'last_obs', 'H'], axis=1)
X_near_to_merge['size_category'] = X_near_to_merge['size_category'].apply(lambda x: x[0:x.index(" ")].strip())







In [3]:
## 6.15% of the target values for pha are True. This makes the target feature unbalanced
pha_counts = df_near['pha'].value_counts()
pha_counts.get(True)/len(df_near)*100

np.float64(6.150529299193334)

In [4]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
import numpy as np



categorical_transformer = Pipeline(
    steps = [
        ("one_hot_encoder", OneHotEncoder(handle_unknown='ignore', drop='first'))
    ]
)

numeric_transformer = Pipeline(
    steps = [
        ('simple_imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
        ('standard_scaler', StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('categorical_transformer', categorical_transformer, make_column_selector(dtype_include=['object','category','str'])),
        ('numeric_transformer', numeric_transformer, make_column_selector(dtype_include='number'))
    ]
)


pipeline_lr = make_pipeline(preprocessor, LogisticRegression(class_weight='balanced'))
pipeline_rfc = make_pipeline(preprocessor, RandomForestClassifier(class_weight='balanced'))
pipeline_gbc = make_pipeline(preprocessor, GradientBoostingClassifier())

pipeline_dtr = make_pipeline(preprocessor, DecisionTreeRegressor())
pipeline_rfr = make_pipeline(preprocessor, RandomForestRegressor())
pipeline_gbr = make_pipeline(preprocessor, GradientBoostingRegressor())
pipeline_linreg = make_pipeline(preprocessor, LinearRegression())



pipelines_classification = {
    "Random Forest Classifier": pipeline_rfc,
    "Logistic Regression": pipeline_lr,
    "Gradient Boosting Classifier" : pipeline_gbc
}

pipelines_regression = {
    "Random Forest Regressor": pipeline_rfr,
    "Decision Tree Regressor": pipeline_dtr,
    "Gradient Boosting Regressor": pipeline_gbr,
    "Linear Regression": pipeline_linreg
}


X_train_pha, X_test_pha, y_train_pha, y_test_pha = train_test_split(X_near_pha, y_pha, test_size=0.2, random_state=42)
X_train_moid, X_test_moid, y_train_moid, y_test_moid = train_test_split(X_near_moid, y_moid, test_size=0.2, random_state=42)

trainsets = {

             "X_near_pha": (X_train_pha, y_train_pha, "cat"), 
             "X_near_moid": (X_train_moid, y_train_moid, "num")

            }

stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

def cross_val_all(trainsets, pipelines_c, pipelines_r):
    for name, (X_train, y_train, type) in trainsets.items():
        print(f"{name} train data cross-validation:\n")
        if name == "X_merged":
            break
        if type == "cat":

            for name, pipeline in pipelines_c.items():

                if y_train.isna().sum() > 0:

                    y_nan_idx = y_train[y_train.isna()].index.tolist()
                    y_train = y_train.drop(y_nan_idx, axis=0).copy()
                    X_train = X_train.drop(y_nan_idx, axis=0).copy()
                    scores = cross_val_score(pipeline, X_train, y_train, cv=stratified_kf, scoring='roc_auc')
                    results[name] = {
                        'model': pipeline, 'cv_auc': scores.mean(),
                        'cv_auc_std': scores.std()
                    }
                    continue

                scores = cross_val_score(pipeline, X_train, y_train, cv=stratified_kf, scoring='roc_auc')
                results[name] = {
                        'model': pipeline, 'cv_auc': scores.mean(),
                        'cv_auc_std': scores.std()
                    }

                print(f"{name:20s}: ROC-AUC score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")
        else:
            
            for name, pipeline in pipelines_r.items():

                if y_train.isna().sum() > 0:

                    y_nan_idx = y_train[y_train.isna()].index.tolist()
                    y_train = y_train.drop(y_nan_idx, axis=0).copy()
                    X_train = X_train.drop(y_nan_idx, axis=0).copy()

                    scores = cross_val_score(pipeline, X_train, y_train, cv=5)
                    print(f"{name:20s}: r2 score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")

                    continue

                scores = cross_val_score(pipeline, X_train, y_train, cv=5)
                print(f"{name:20s}: r2 score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")

        print("====================")


cross_val_all(trainsets, pipelines_classification, pipelines_regression)




X_near_pha train data cross-validation:

Random Forest Classifier: ROC-AUC score = 0.999 ± 0.001
Logistic Regression : ROC-AUC score = 0.998 ± 0.001
Gradient Boosting Classifier: ROC-AUC score = 0.999 ± 0.001
X_near_moid train data cross-validation:

Random Forest Regressor: r2 score = 0.812 ± 0.007
Decision Tree Regressor: r2 score = 0.634 ± 0.013
Gradient Boosting Regressor: r2 score = 0.799 ± 0.005
Linear Regression   : r2 score = 0.504 ± 0.166


In [5]:
results

{'Random Forest Classifier': {'model': Pipeline(steps=[('columntransformer',
                   ColumnTransformer(transformers=[('categorical_transformer',
                                                    Pipeline(steps=[('one_hot_encoder',
                                                                     OneHotEncoder(drop='first',
                                                                                   handle_unknown='ignore'))]),
                                                    <sklearn.compose._column_transformer.make_column_selector object at 0x7fbc75a3e510>),
                                                   ('numeric_transformer',
                                                    Pipeline(steps=[('simple_imputer',
                                                                     SimpleImputer()),
                                                                    ('standard_scaler',
                                                                     Sta

In [6]:
from pathlib import Path

DATA_PATH = Path("../artifacts/data")
DATA_PATH.mkdir(parents=True, exist_ok=True)

numeric_columns = X_train_pha.select_dtypes(include='number')
cat_columns = X_train_pha.select_dtypes(include=['category', 'str', 'object'])
quantiles_x_train = X_train_pha.quantile([.1, .25, .5, .75, .9], axis=0, numeric_only=True)
X_train_pha.describe()

lst = numeric_columns.std().index
vals = numeric_columns.std().values.tolist()

X_train_stats = pd.concat([pd.DataFrame([vals], columns=lst, index=['std']),quantiles_x_train,\
            pd.DataFrame([numeric_columns.min().values.tolist()], columns=lst, index=['min']),\
                         pd.DataFrame([numeric_columns.max().values.tolist()], columns=lst, index=['max'])])
#dftemp = pd.DataFrame(np.array(numeric_columns.std().values).reshape(-1), columns=lst)
X_train_stats.round(3).to_csv(DATA_PATH / "X_train_stats.csv")

#pd.concat([quantiles_x_train, numeric_columns.std()], ignore_index=True, axis=1)


In [7]:
from sklearn.metrics import f1_score, recall_score


best_model_pha = pipelines_classification['Random Forest Classifier']
best_model_pha.fit(X_train_pha, y_train_pha)

y_train_pha_pred = best_model_pha.predict(X_train_pha)
y_train_pha_proba = best_model_pha.predict_proba(X_train_pha)[:, 1]


best_f1 = 0
print(f"Train f1_score: {f1_score(y_train_pha, y_train_pha_pred)}")

thresholds = np.arange(0.1, 1, 0.1)

for thresh in thresholds:
    hold = [1 if thresh > y_train_pha_proba[i] else 0 for i in range(len(y_train_pha_proba))]
    temp_f1 = f1_score(y_train_pha, hold)
    print(temp_f1)
    if temp_f1 > best_f1:
        best_f1 = temp_f1



Train f1_score: 1.0
0.0
0.0
0.0
0.0
0.0
0.00024221872350732712
0.0010290245452619473
0.002177858439201452
0.0038684719535783366


In [8]:

from sklearn.metrics import f1_score, recall_score
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# Best model for PHA

best_model_pha = pipelines_classification["Random Forest Classifier"]

best_model_pha.fit(X_train_pha, y_train_pha)

y_test_pha_pred = best_model_pha.predict(X_test_pha)
y_test_pha_proba = best_model_pha.predict_proba(X_test_pha)[:, 1]


print(y_test_pha.sum()/len(y_test_pha))
print(y_test_pha_pred.sum()/len(y_test_pha_pred))
print(classification_report(y_test_pha, y_test_pha_pred))
print(roc_auc_score(y_test_pha_pred, y_test_pha_proba))

columntransformer_pha = best_model_pha['columntransformer']
columntransformer_pha.fit(X_train_pha)

joblib.dump(best_model_pha['randomforestclassifier'], '../artifacts/best_model_pha.pkl')
joblib.dump(columntransformer_pha, '../artifacts/columntransformer_pha.pkl')

#Saving reference train sets for monitoring
joblib.dump(X_train_pha, '../artifacts/X_train_pha.pkl')
joblib.dump(y_train_pha, '../artifacts/y_train_pha.pkl')

#Saving test sets for simulation of new data during monitoring
joblib.dump(X_test_pha, '../artifacts/X_test_pha.pkl')
joblib.dump(y_test_pha, '../artifacts/y_test_pha.pkl')


0.0634613055589197
0.06406685236768803
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      7733
        True       0.98      0.99      0.98       524

    accuracy                           1.00      8257
   macro avg       0.99      0.99      0.99      8257
weighted avg       1.00      1.00      1.00      8257

1.0


['../artifacts/y_test_pha.pkl']

In [9]:


moid_gbr_param_grid = {
    "gradientboostingregressor__n_estimators": [10, 30, 100, 300],
    "gradientboostingregressor__min_samples_leaf": [1, 2, 3],
    "gradientboostingregressor__max_depth": [2, 3, 5, 9]
}

moid_gbr_grid = GridSearchCV(
    pipelines_regression["Gradient Boosting Regressor"],
    moid_gbr_param_grid,
    cv=5,
    scoring="r2",
    n_jobs = -1,
    return_train_score=True
)



In [ ]:

#There are null values in y_train_moid, hence the next three lines, which drop these rows before we're able to fit moid_gbr_grid 
# y_train_na_idx = y_train_moid[y_train_moid.isna()].index.tolist()
# y_train_moid.drop(y_train_na_idx, axis=0, inplace=True)
# X_train_no_moid.drop(y_train_na_idx, axis=0, inplace=True)

# moid_gbr_grid.fit(X_train_no_moid, y_train_moid)
# moid_best_model = moid_gbr_grid.best_estimator_

# y_na_idx = y_test_moid[y_test_moid.isna()].index.tolist()
# y_test_moid.drop(y_na_idx, axis=0, inplace=True)
# X_test_no_moid.drop(y_na_idx, axis=0, inplace=True)

# y_pred_moid = moid_best_model.predict(X_test_no_moid)



# print(f"GBR ---- R2 score: {r2_score(y_pred_moid, y_test_moid)}")


gbr = pipelines_regression["Gradient Boosting Regressor"]

gbr.set_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('categorical_transformer',
                                    Pipeline(steps=[('one_hot_encoder',
                                                     OneHotEncoder(drop='first',
                                                                   handle_unknown='ignore'))]),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x7fbc75a3e510>),
                                   ('numeric_transformer',
                                    Pipeline(steps=[('simple_imputer',
                                                     SimpleImputer()),
                                                    ('standard_scaler',
                                                     StandardScaler())]),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x7fbc75aa8550>)])),
  ('gradientboostingregressor', GradientBoos

In [28]:
from sklearn.metrics import r2_score
y_train_na_idx = y_train_moid[y_train_moid.isna()].index.tolist()
y_train_moid.drop(y_train_na_idx, axis=0, inplace=True)
X_train_moid.drop(y_train_na_idx, axis=0, inplace=True)

gbr.set_params(gradientboostingregressor__n_estimators=5000, gradientboostingregressor__n_iter_no_change=1000,\
               gradientboostingregressor__min_impurity_decrease=0.04,gradientboostingregressor__min_samples_split=10,\
                gradientboostingregressor__learning_rate=0.02)
gbr.fit(X_train_moid, y_train_moid)

moid_pred = gbr.predict(X_test_moid)

r2_score(y_test_moid, moid_pred)

0.8017926851620524

In [42]:
numeric_columnsdf = df_near.select_dtypes(include="number").columns.tolist()
df_near[numeric_columns].corr()

numeric_columnsmoid = X_near_moid.select_dtypes(include="number").columns.tolist()
X_near_moid[numeric_columnsmoid].corr()
X_near_moid["data_arc"].describe()
X_near_moid[X_near_moid["data_arc"] > 800][numeric_columnsmoid].corr()


,H,diameter_km,eccentricity,semimajor_axis_au,inclination_deg,perihelion_distance_au,aphelion_distance_au,orbital_period_days,mean_motion_deg_day,condition_code,data_arc
H,1.000000,-0.622317,-0.373319,-0.404489,-0.361615,-0.052958,-0.423486,-0.397773,0.363143,0.221998,-0.530192
diameter_km,-0.622317,1.000000,0.199803,0.248907,0.199902,0.034964,0.260008,0.252971,-0.192999,-0.133225,0.499790
eccentricity,-0.373319,0.199803,1.000000,0.540214,0.020089,-0.485002,0.709410,0.541456,-0.415849,-0.078036,0.023944
semimajor_axis_au,-0.404489,0.248907,0.540214,1.000000,-0.028789,0.420983,0.972080,0.994987,-0.880960,-0.141894,0.024766
inclination_deg,-0.361615,0.199902,0.020089,-0.028789,1.000000,-0.055152,-0.016864,-0.026189,0.018761,-0.012183,0.052269
perihelion_distance_au,-0.052958,0.034964,-0.485002,0.420983,-0.055152,1.000000,0.196399,0.396666,-0.494674,-0.071393,0.026479
aphelion_distance_au,-0.423486,0.260008,0.709410,0.972080,-0.016864,0.196399,1.000000,0.972952,-0.824339,-0.134929,0.019858
orbital_period_days,-0.397773,0.252971,0.541456,0.994987,-0.026189,0.396666,0.972952,1.000000,-0.835665,-0.132503,0.022560
mean_motion_deg_day,0.363143,-0.192999,-0.415849,-0.880960,0.018761,-0.494674,-0.824339,-0.835665,1.000000,0.175770,-0.034888
condition_code,0.221998,-0.133225,-0.078036,-0.141894,-0.012183,-0.071393,-0.134929,-0.132503,0.175770,1.000000,-0.305373


In [ ]:

y_train_na_idx = y_train_moid[y_train_moid.isna()].index.tolist()
y_train_moid.drop(y_train_na_idx, axis=0, inplace=True)
X_train_moid.drop(y_train_na_idx, axis=0, inplace=True)

best_model_moid = pipelines_regression['Random Forest Regressor']
best_model_moid.fit(X_train_moid, y_train_moid)

joblib.dump(best_model_moid['randomforestregressor'], '../artifacts/best_model_moid.pkl')

columntransformer_moid = best_model_moid['columntransformer']
columntransformer_moid.fit(X_train_moid)

joblib.dump(columntransformer_moid, '../artifacts/columntransformer_moid.pkl')


#Saving reference train sets for monitoring
joblib.dump(X_train_moid, '../artifacts/X_train_moid.pkl')
joblib.dump(y_train_moid, '../artifacts/y_train_moid.pkl')

#Saving test sets for simulation of new data during monitoring
joblib.dump(X_test_moid, '../artifacts/X_test_moid.pkl')
joblib.dump(y_test_moid, '../artifacts/y_test_moid.pkl')

['../artifacts/y_test_moid.pkl']

In [ ]:

X_train_moid_stats = X_train_moid.describe()
joblib.dump(X_train_moid_stats, "../artifacts/X_train_moid_stats.pkl")


['../artifacts/X_train_moid_stats.pkl']

In [ ]:
import os
from google.cloud import storage
storage.blob._MAX_MULTIPART_SIZE = 5 * 1024* 1024


def upload_local_directory(bucket_name, local_folder_path, gcs_folder_path=None):
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    for root, dirs, files in os.walk(local_folder_path):
        for file in files:
            local_file_path = os.path.join(root, file)

            relative_path = os.path.relpath(local_file_path, local_folder_path)
            
            gcs_destination = os.path.join(gcs_folder_path, relative_path ).replace("\\", "/")
            
            blob = bucket.blob(gcs_destination)
            blob._chunk_size = 5 * 1024* 1024
            blob.upload_from_filename(local_file_path)
           



upload_local_directory('project-3e6b348d-e2ae-4a47-9af_cloudbuild', '../artifacts','artifacts/')
#upload_local_directory('project-3e6b348d-e2ae-4a47-9af_cloudbuild', '../artifacts/data','artifacts/')
    

In [ ]:
import xgboost

